# Analysis of the pgchecker runs

Condenses the runs of `main.ipynb` into the three views of Hunold et al.,
*Parallel Computing* 129 (2026) 103205:

1. **Best/worst overhead heatmap** (Fig. 5): one colour-coded cell per collective and
   message size, for the best and the worst observed overhead.
2. **Runtime per message size** (Fig. 8): MPI library default vs. the fastest
   guideline implementation, per collective.
3. **Violation share** (Table 2): share of measurements with a high or very high
   overhead, per collective.

## The metric (paper, section 5.1.4)

Per execution, collective and message size:

* `eta_L` = median runtime of the **MPI library implementation** (`default`),
* `eta_M` = median runtime of the **fastest mock-up** (guideline implementation),
* `o = max(0, eta_L / eta_M - 1)` (overhead).

`o = 0.3` means the library function was 30 % slower than the fastest guideline
implementation. Every run `(k, i)` counts as one *execution*.

## Settings (edit here)

**This is the only section you need to edit.** By default the notebook analyses the
**newest** experiment under `results/`. To pick another one, or to pool several, set
`EXPERIMENT_FILTER` to (part of) their `experiment_id`, e.g. `"pgchecker-uc-2nodes"`.

In [ ]:
# None = newest experiment under results/; a string pools every experiment whose
# experiment_id contains it, e.g. "pgchecker-uc-2nodes" or "pgchecker-tacc-2nodes".
EXPERIMENT_FILTER = None

# Write the figures and the table to results/analysis/<experiment>/.
SAVE_OUTPUT = False

## Setup

From here on, every cell runs as-is. This one loads the libraries and the plot style.

In [ ]:
from pathlib import Path
import json
import re

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from matplotlib.colors import ListedColormap
from matplotlib.patches import Patch
from matplotlib.ticker import FuncFormatter, NullFormatter

RESULTS_ROOT = Path.cwd() / "results"

# Overhead categories of the paper; colours read out of Fig. 5.
SEVERITY = [
    (r"($\leq$ 10%) low",      "#68b69c"),
    (r"($\leq$ 50%) medium",   "#eea8a8"),
    (r"($\leq$ 200%) high",    "#d46464"),
    (r"($>$ 200%) very high",  "#ca3434"),
]
SEVERITY_LABELS = [row[0] for row in SEVERITY]
SEVERITY_COLORS = [row[1] for row in SEVERITY]
HIGH_THRESHOLD = 0.5        # o > 0.5 = high or very high (Table 2)

MISSING_COLOR = "#ffffff"
LINE_COLOR = "#000000"
LINE_WIDTH = 0.61
CELL_ASPECT = 15.9 / 23.2   # cell shape of Fig. 5
CELL_WIDTH_IN = 0.72
DEFAULT_COLOR, GUIDELINE_COLOR = "#d95f02", "#1b9e77"   # Fig. 8 (ColorBrewer Dark2)

plt.rcParams.update({
    "figure.facecolor": "white",
    "axes.facecolor": "white",
    "savefig.facecolor": "white",
    "font.family": "serif",
    "mathtext.fontset": "dejavuserif",
    "font.size": 10,
})


def severity_index(overhead: float) -> int:
    # Colour category of an overhead value; -1 means no data.
    if not np.isfinite(overhead):
        return -1
    if overhead <= 0.1:
        return 0
    if overhead <= 0.5:
        return 1
    if overhead <= 2.0:
        return 2
    return 3


def format_size(num_bytes: float) -> str:
    # 80000 -> '80 000' (thin space as the group separator, as in the paper)
    return f"{int(num_bytes):,}".replace(",", " ")


def format_overhead(overhead: float) -> str:
    return "" if not np.isfinite(overhead) else f"{overhead * 100.0:.0f}"


def load_json(path: Path) -> dict:
    try:
        return json.loads(path.read_text(encoding="utf-8"))
    except (FileNotFoundError, json.JSONDecodeError):
        return {}


def save_figure(fig, filename: str) -> None:
    if SAVE_OUTPUT:
        OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
        fig.savefig(OUTPUT_DIR / filename, dpi=200, bbox_inches="tight")
        print("Saved:", OUTPUT_DIR / filename)

## 1. Find the runs

`main.ipynb` writes every experiment to `results/{experiment_id}/`, with
`experiment_id = <experiment name>-<UTC timestamp>`:

```
results/{experiment_id}/
  experiment.json  config_resolved.yaml
  k{k}/                      # one allocation per k
    allocation.json          # machines used by this allocation
    i{i}/                    # one run of the command per i
      run.json  run.log      # metadata (exit_code, nodes, ...) and full output
      pgmpi_info.csv         # collective/algorithm of every data file
      data_<MPI_Call>_<algorithm>.dat   # raw runtimes
```

Runs with a non-zero `exit_code` or without `.dat` files are skipped.

In [ ]:
TIMESTAMP = re.compile(r"\d{8}T\d{6}Z")


def experiment_dirs(filter_string):
    if not RESULTS_ROOT.is_dir():
        raise FileNotFoundError(f"No results/ directory in {Path.cwd()} -- run main.ipynb first.")
    candidates = [path for path in sorted(RESULTS_ROOT.iterdir())
                  if path.is_dir() and (path / "experiment.json").exists()]
    if not candidates:
        raise FileNotFoundError(f"No experiment in {RESULTS_ROOT} -- run main.ipynb first.")
    if filter_string is None:
        # Newest by the timestamp in the experiment_id.
        def stamp(path):
            match = TIMESTAMP.search(path.name)
            return match.group(0) if match else ""
        return [max(candidates, key=stamp)]
    matching = [path for path in candidates if filter_string in path.name]
    if not matching:
        raise FileNotFoundError(f"No experiment contains '{filter_string}'. "
                                f"Available: {[path.name for path in candidates]}")
    return matching


rows = []
for exp_index, exp_dir in enumerate(experiment_dirs(EXPERIMENT_FILTER)):
    exp_tag = chr(ord("A") + exp_index) if exp_index < 26 else f"E{exp_index}"
    for k_dir in sorted(path for path in exp_dir.glob("k*") if path.is_dir()):
        allocation = load_json(k_dir / "allocation.json")
        machines = [node.get("name") for node in allocation.get("selected_nodes", [])
                    if node.get("name")]
        for i_dir in sorted(path for path in k_dir.glob("i*") if path.is_dir()):
            run = load_json(i_dir / "run.json")
            rows.append({
                "execution": f"{exp_dir.name}/{k_dir.name}/{i_dir.name}",
                "label": f"{exp_tag} {k_dir.name}/{i_dir.name}",
                "experiment_id": exp_dir.name,
                "path": i_dir,
                "exit_code": run.get("exit_code"),
                "node_count": run.get("node_count"),
                "processes_per_node": run.get("processes_per_node"),
                "machines": ", ".join(machines),
                "dat_files": len(list(i_dir.glob("data_*.dat"))),
            })

found = pd.DataFrame(rows)
if found.empty:
    raise RuntimeError("The selected experiment has no runs (k*/i*/).")
found["usable"] = (found["dat_files"] > 0) & found["exit_code"].fillna(-1).eq(0)
executions = found[found["usable"]].reset_index(drop=True)
for row in found[~found["usable"]].itertuples(index=False):
    print(f"Skipped {row.execution}: exit_code={row.exit_code}, dat_files={row.dat_files}")
if executions.empty:
    raise RuntimeError("No usable run (exit_code 0 with .dat files) in the selected experiment.")

EXPERIMENT_NAME = EXPERIMENT_FILTER or executions["experiment_id"].iloc[0]
OUTPUT_DIR = RESULTS_ROOT / "analysis" / EXPERIMENT_NAME
print(f"Experiment: {EXPERIMENT_NAME}   usable runs: {len(executions)} of {len(found)}")
display(executions[["label", "experiment_id", "node_count", "processes_per_node",
                    "dat_files", "machines"]])

## 2. Median runtimes and overhead `o`

The medians are computed from the `.dat` raw data (median over all repetitions per
message size). `pgmpi_info.csv` maps every file to its collective and algorithm, since
MPI names contain underscores themselves (`MPI_Reduce_scatter_block`).

In [ ]:
KNOWN_COLLECTIVES = sorted([
    "MPI_Allgather", "MPI_Allreduce", "MPI_Alltoall", "MPI_Bcast", "MPI_Gather",
    "MPI_Reduce", "MPI_Reduce_scatter_block", "MPI_Scan", "MPI_Scatter",
], key=len, reverse=True)
KEYS = ["execution", "collective", "message_size"]


def algorithm_files(run_dir: Path) -> dict:
    # {(collective, algorithm): path}; name-based fallback without pgmpi_info.csv.
    mapping = {}
    info_path = run_dir / "pgmpi_info.csv"
    if info_path.exists():
        info = pd.read_csv(info_path, sep=";", comment="#")
        for collective, algorithm in info[["mpiname", "algname"]].itertuples(index=False):
            path = run_dir / f"data_{collective}_{algorithm}.dat"
            if path.exists():
                mapping[(collective, algorithm)] = path
    mapped = set(mapping.values())
    for path in sorted(run_dir.glob("data_*.dat")):
        if path in mapped:
            continue
        stem = path.stem[len("data_"):]
        collective = next((name for name in KNOWN_COLLECTIVES if stem.startswith(name + "_")), None)
        if collective:
            mapping[(collective, stem[len(collective) + 1:])] = path
    return mapping


chunks = []
for row in executions.itertuples(index=False):
    for (collective, algorithm), path in sorted(algorithm_files(row.path).items()):
        frame = pd.read_csv(path, sep=r"\s+", comment="#")
        medians_ms = (frame.groupby("count")["runtime_sec"].median() * 1000.0).rename("median_ms")
        chunk = medians_ms.reset_index().rename(columns={"count": "message_size"})
        chunk["execution"], chunk["label"] = row.execution, row.label
        chunk["collective"], chunk["algorithm"] = collective, algorithm
        chunks.append(chunk)
medians = pd.concat(chunks, ignore_index=True)

library = medians[medians["algorithm"] == "default"]
mockups = medians[medians["algorithm"] != "default"]
fastest = mockups.loc[mockups.groupby(KEYS)["median_ms"].idxmin()]
overhead = (
    library[[*KEYS, "label", "median_ms"]].rename(columns={"median_ms": "eta_lib_ms"})
    .merge(fastest[[*KEYS, "median_ms", "algorithm"]]
           .rename(columns={"median_ms": "eta_mock_ms", "algorithm": "fastest_mockup"}),
           on=KEYS, how="left")
)
overhead["overhead"] = (overhead["eta_lib_ms"] / overhead["eta_mock_ms"] - 1.0).clip(lower=0.0)

COLLECTIVES = sorted(overhead["collective"].unique())
MESSAGE_SIZES = sorted(overhead["message_size"].unique())
nodes = int(executions["node_count"].iloc[0])
ppn = int(executions["processes_per_node"].iloc[0])
CAPTION = (f"{EXPERIMENT_NAME}; {nodes * ppn} ({nodes} x {ppn}) processes, "
           f"{overhead['execution'].nunique()} executions")
print(f"{len(COLLECTIVES)} collectives x {len(MESSAGE_SIZES)} message sizes x "
      f"{overhead['execution'].nunique()} executions")

## 3. Best/worst overhead heatmap (Fig. 5)

Per collective and message size, the lowest (left) and highest (right) overhead over all
executions; the number in a cell is `o` in percent. Best and worst of one cell may come
from different executions.

In [ ]:
grouped = overhead.groupby(["collective", "message_size"])["overhead"]
panels = [("Best", overhead.loc[grouped.idxmin()]), ("Worst", overhead.loc[grouped.idxmax()])]

cmap = ListedColormap(SEVERITY_COLORS)
cmap.set_bad(MISSING_COLOR)
width = CELL_WIDTH_IN * len(MESSAGE_SIZES) + 2.6
height = CELL_WIDTH_IN * CELL_ASPECT * len(COLLECTIVES) + 2.2
fig, axes = plt.subplots(1, 2, figsize=(2 * width, height), squeeze=False, layout="constrained")

for ax, (title, frame) in zip(axes[0], panels):
    values = (frame.pivot(index="collective", columns="message_size", values="overhead")
              .reindex(index=COLLECTIVES, columns=MESSAGE_SIZES).to_numpy(float))
    severities = np.vectorize(severity_index)(values).astype(float)
    ax.imshow(np.ma.masked_where(severities < 0, severities), cmap=cmap,
              vmin=-0.5, vmax=len(SEVERITY_COLORS) - 0.5, aspect=CELL_ASPECT)
    ax.set_title(title, pad=10)
    ax.set_xticks(range(len(MESSAGE_SIZES)), [format_size(s) for s in MESSAGE_SIZES],
                  rotation=45, ha="right")
    ax.set_yticks(range(len(COLLECTIVES)), COLLECTIVES, rotation=45, ha="right",
                  fontfamily="monospace")
    # The minor grid draws the cell borders.
    ax.set_xticks(np.arange(-0.5, len(MESSAGE_SIZES), 1), minor=True)
    ax.set_yticks(np.arange(-0.5, len(COLLECTIVES), 1), minor=True)
    ax.grid(which="minor", color=LINE_COLOR, linewidth=LINE_WIDTH)
    ax.tick_params(which="minor", length=0)
    ax.set_xlabel("Message Size [Byte]")
    for r in range(len(COLLECTIVES)):
        for c in range(len(MESSAGE_SIZES)):
            ax.text(c, r, format_overhead(values[r, c]), ha="center", va="center", fontsize=10)

fig.legend(handles=[Patch(facecolor=color, edgecolor=LINE_COLOR, linewidth=LINE_WIDTH, label=label)
                    for label, color in zip(SEVERITY_LABELS, SEVERITY_COLORS)],
           loc="outside lower center", ncol=len(SEVERITY_LABELS), fontsize=9,
           title=CAPTION, title_fontsize=10)
save_figure(fig, "overhead_best_worst.png")
plt.show()

## 4. Runtime per message size (Fig. 8)

Per collective: the MPI library default (`eta_L`) and the fastest guideline
implementation (`eta_M`). The line is the median over all executions, the shaded band
spans the fastest to the slowest execution. Where the orange line lies above the green
one, the library violates a performance guideline; the gap is the overhead `o`.

In [ ]:
columns = min(3, len(COLLECTIVES))
rows_n = int(np.ceil(len(COLLECTIVES) / columns))
fig, axes = plt.subplots(rows_n, columns, figsize=(5.0 * columns, 3.4 * rows_n),
                         squeeze=False, layout="constrained")

for ax, collective in zip(axes.flat, COLLECTIVES):
    subset = overhead[overhead["collective"] == collective]
    sizes = sorted(subset["message_size"].unique())
    x = np.arange(len(sizes))
    for column, label, color, marker in [
        ("eta_lib_ms", "Default Algorithm (autoselect)", DEFAULT_COLOR, "o"),
        ("eta_mock_ms", "Guideline-Based Algorithm", GUIDELINE_COLOR, "s"),
    ]:
        stats = subset.groupby("message_size")[column].agg(["median", "min", "max"]).reindex(sizes)
        ax.fill_between(x, stats["min"], stats["max"], color=color, alpha=0.18, linewidth=0)
        ax.plot(x, stats["median"], color=color, marker=marker, markersize=4,
                linewidth=1.2, label=label)
    ax.set_yscale("log")
    ax.yaxis.set_major_formatter(FuncFormatter(lambda value, _: f"{value:g}"))
    ax.yaxis.set_minor_formatter(NullFormatter())
    ax.set_xticks(x, [format_size(s) for s in sizes], rotation=45, ha="right")
    ax.set_title(collective, fontsize=10, fontfamily="monospace")
    ax.set_xlabel("Message Size [Byte]")
    ax.set_ylabel("Execution Time [ms]")
    ax.grid(axis="y", color="#c8c8c8", linewidth=LINE_WIDTH)

for ax in list(axes.flat)[len(COLLECTIVES):]:
    ax.set_visible(False)
handles, labels = axes.flat[0].get_legend_handles_labels()
fig.legend(handles, labels, loc="outside upper center", ncol=2, fontsize=9)
fig.supxlabel(CAPTION, fontsize=10)
save_figure(fig, "runtime_default_vs_guideline.png")
plt.show()

## 5. Violation share per collective (Table 2)

Share of all measurements (message sizes x executions) with a high or very high
overhead (`o > 0.5`); lower is better.

In [ ]:
violated = overhead.assign(violated=overhead["overhead"] > HIGH_THRESHOLD)
table2 = (
    violated.groupby("collective")
    .agg(measurements=("violated", "size"), violations=("violated", "sum"),
         share=("violated", "mean"))
)
table2["share"] = (table2["share"] * 100).round(0).astype(int).astype(str) + " %"
table2.index.name = "MPI call"
display(table2)

if SAVE_OUTPUT:
    OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
    table2.to_csv(OUTPUT_DIR / "violation_share.csv")
    overhead.drop(columns="label").to_csv(OUTPUT_DIR / "overhead.csv", index=False)
    print("Saved:", OUTPUT_DIR / "violation_share.csv", "|", OUTPUT_DIR / "overhead.csv")